# Behavior overview

Session-1 behavior across all study participants of `ds-abstractvalue`.

Each trial:
- A Gabor is shown with one of many orientations in `[0, 180)°`.
- The orientation maps to a CHF value in `[0, 42]` (`cdf` or `inverse_cdf`, counter-balanced across subjects).
- The subject reports their estimated value on a response bar (BDM bid).

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from abstract_values.behavior.data import Subject, get_all_subjects, get_all_subject_ids, get_all_behavioral_data

plt.rcParams.update({
    "font.size": 14, "axes.titlesize": 16, "axes.labelsize": 14,
    "xtick.labelsize": 12, "ytick.labelsize": 12, "legend.fontsize": 12,
    "figure.titlesize": 17,
})

VALUE_RANGE = (0, 42)   # CHF
ORI_RANGE = (0, 180)    # degrees

# Bedi et al. 2026 condition palette (dark green / dark orange, no Uniform)
PALETTE = {"CDF": "#E76F51", "Inverse CDF": "#2A9D8F"}
HUE_ORDER = ["CDF", "Inverse CDF"]

## Load data

In [ ]:
# Frame-1 slider confirms (RT ~17 ms) are already blanked upstream by the
# Subject machinery -- see abstract_values.utils.data.MIN_VALID_RT for why.
# Pass min_rt=None here to get the raw, unfiltered bids back.
df = get_all_behavioral_data()
df = df[df["event_type"] == "feedback"].copy()
df["response"] = pd.to_numeric(df["response"], errors="coerce")
df = df.reset_index()
df["error"] = df["response"] - df["value"]
df["abs_error"] = df["error"].abs()
df["Mapping"] = df["mapping"].map({"cdf": "CDF", "inverse_cdf": "Inverse CDF"})

n_bad = int(df["invalid_response"].sum())
print(f"Frame-1 responses blanked upstream: {n_bad} in subject(s) "
      f"{sorted(df.loc[df['invalid_response'], 'subject'].unique())}")

# Veridical (true) mapping per (Mapping, orientation), deterministic from the data
truth = df.groupby(["Mapping", "orientation"], as_index=False)["value"].first()

df.head()

## Per-subject summary

One row per (subject, mapping condition) with headline numbers: mean absolute response error, mean RT, and non-response rate.

In [ ]:
summary = (df.assign(no_response=df["response"].isna())
             .groupby(["subject", "Mapping"])
             .agg(
                 n_trials=("response", "size"),
                 non_response_rate=("no_response", "mean"),
                 mean_abs_error=("abs_error", "mean"),
                 median_abs_error=("abs_error", "median"),
                 mean_rt=("rt", "mean"),
             )
             .reset_index())

# Within-condition quantile (percentile rank): fraction of subjects in the SAME
# mapping condition whose value is <= this subject's. Higher = worse (more error /
# more misses) relative to peers in that condition.
summary["abs_error_q"] = summary.groupby("Mapping")["mean_abs_error"].rank(pct=True)
summary["non_response_q"] = summary.groupby("Mapping")["non_response_rate"].rank(pct=True)

summary = summary.set_index(["subject", "Mapping"]).sort_index()

(summary.style
    .format({
        "n_trials": "{:d}",
        "non_response_rate": "{:.1%}",
        "mean_abs_error": "{:.2f} CHF",
        "median_abs_error": "{:.2f} CHF",
        "mean_rt": "{:.2f} s",
        "abs_error_q": "{:.0%}",
        "non_response_q": "{:.0%}",
    })
    .background_gradient(subset=["abs_error_q", "non_response_q"], cmap="Reds", vmin=0, vmax=1)
    .set_caption("Per-subject behavioral summary by mapping condition. "
                 "*_q columns = within-condition percentile rank (higher = worse vs. peers)."))

### Distribution of per-subject error & misses within condition

Where each participant sits relative to peers in the **same** mapping condition. Boxes show the within-condition quartiles (so a subject above the upper whisker is in the top quantile of error / misses); dots are individual subjects.

In [ ]:
summary_long = summary.reset_index()

metrics = [("mean_abs_error", "Mean abs error (CHF)"),
           ("non_response_rate", "Non-response rate")]

with sns.plotting_context("talk"), sns.axes_style("ticks"):
    fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.3), constrained_layout=True)
    for ax, (col, label) in zip(axes, metrics):
        sns.boxplot(data=summary_long, x="Mapping", y=col, order=HUE_ORDER,
                    hue="Mapping", hue_order=HUE_ORDER, palette=PALETTE,
                    width=0.5, showfliers=False, boxprops=dict(alpha=0.25),
                    legend=False, ax=ax)
        sns.stripplot(data=summary_long, x="Mapping", y=col, order=HUE_ORDER,
                      hue="Mapping", hue_order=HUE_ORDER, palette=PALETTE,
                      size=7, jitter=0.18, edgecolor="white", linewidth=0.6,
                      legend=False, ax=ax)
        ax.set_xlabel("")
        ax.set_ylabel(label)
        if col == "non_response_rate":
            ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))
        sns.despine(ax=ax, offset=5, trim=True)
    fig.suptitle("Per-subject error & misses within each condition (box = quartiles)", y=1.05)

## Mean estimate by orientation (group level)

Across subjects, mean reported estimate (CHF) as a function of stimulus orientation, for each mapping condition. Shaded bands are ±1 SEM across subjects. Dashed lines show the veridical (true) mapping for each condition.

In [ ]:
mean_resp = (df.groupby(["subject", "Mapping", "orientation"], as_index=False)["response"]
                .mean())

endpoints = (mean_resp[mean_resp["orientation"] == mean_resp["orientation"].max()]
             .groupby("Mapping")["response"].mean())

with sns.plotting_context("talk"), sns.axes_style("ticks"):
    fig, ax = plt.subplots(figsize=(4.6, 3.4), constrained_layout=True)

    # Linear-identity reference (black, behind everything)
    ax.axline((ORI_RANGE[0], VALUE_RANGE[0]), (ORI_RANGE[1], VALUE_RANGE[1]),
              color="k", ls=":", lw=0.8, alpha=0.5, zorder=-1)

    # Veridical mapping per condition (dashed, behind data)
    for m in HUE_ORDER:
        sub = truth[truth["Mapping"] == m].sort_values("orientation")
        ax.plot(sub["orientation"], sub["value"],
                color=PALETTE[m], ls="--", lw=1.2, alpha=0.5, zorder=0)

    sns.lineplot(
        data=mean_resp, x="orientation", y="response",
        hue="Mapping", hue_order=HUE_ORDER, palette=PALETTE,
        errorbar=("se", 1), legend=False, ax=ax,
    )

    x_label = mean_resp["orientation"].max() + 4
    ax.text(x_label, endpoints["CDF"],         "CDF",
            color=PALETTE["CDF"],         fontsize=11, ha="left", va="center", fontweight="bold")
    ax.text(x_label, endpoints["Inverse CDF"], "Inverse CDF",
            color=PALETTE["Inverse CDF"], fontsize=11, ha="left", va="center", fontweight="bold")

    ax.set_xlim(0, 210)
    ax.set_ylim(VALUE_RANGE)
    ax.set_xticks([0, 45, 90, 135, 180])
    ax.set_yticks([0, 10, 20, 30, 42])
    ax.set_xlabel("Orientation (°)")
    ax.set_ylabel("Mean estimate (CHF)")

    sns.despine(ax=ax, offset=5, trim=True)

## RT distributions per subject

In [ ]:
g = sns.FacetGrid(df, col="subject", hue="Mapping", hue_order=HUE_ORDER, palette=PALETTE,
                  col_wrap=4, margin_titles=True, height=2.8)
g.map(sns.histplot, "rt", bins=np.arange(0, 3.5, 0.1), kde=True)
g.set_axis_labels("RT (s)", "Count")
g.add_legend()
g.figure.suptitle("RT distributions per subject", y=1.03)

## Mean response as a function of orientation

In [ ]:
mean_resp = (df.groupby(["subject", "Mapping", "orientation"], as_index=False)["response"]
                .mean())

g = sns.relplot(data=mean_resp, x="orientation", y="response",
                hue="Mapping", hue_order=HUE_ORDER, palette=PALETTE,
                kind="line", col="subject", col_wrap=4, height=2.8)

# Add identity line + veridical mapping per condition to every panel
for ax in g.axes.flat:
    ax.axline((ORI_RANGE[0], VALUE_RANGE[0]), (ORI_RANGE[1], VALUE_RANGE[1]),
              color="k", ls=":", lw=0.7, alpha=0.5, zorder=-1)
    for m in HUE_ORDER:
        sub = truth[truth["Mapping"] == m].sort_values("orientation")
        ax.plot(sub["orientation"], sub["value"],
                color=PALETTE[m], ls="--", lw=1.0, alpha=0.5, zorder=0)

g.set_axis_labels("Orientation (°)", "Mean response (CHF)")
g.figure.suptitle("Mean response by orientation, per subject", y=1.03)

## Signed error as a function of orientation

Smoothed with a centered rolling window (3 orientations) within subject × mapping.

In [ ]:
err = (df.groupby(["subject", "Mapping", "orientation"], as_index=False)["error"]
          .mean()
          .sort_values(["subject", "Mapping", "orientation"]))
err["error_smooth"] = (err.groupby(["subject", "Mapping"])["error"]
                          .transform(lambda s: s.rolling(3, center=True, min_periods=1).mean()))

# Group-level
g = sns.relplot(data=err, x="orientation", y="error_smooth",
                hue="Mapping", hue_order=HUE_ORDER, palette=PALETTE,
                kind="line", errorbar="se", height=4, aspect=1.4)
g.set_axis_labels("Orientation (°)", "Mean signed error (CHF)")
plt.axhline(0, color="k", linestyle="--", alpha=0.5)
for ori in (45, 90, 135):
    plt.axvline(ori, color="grey", linestyle=":", alpha=0.5)
g.figure.suptitle("Mean signed error by orientation (across subjects)", y=1.02)

In [ ]:
g = sns.relplot(data=err, x="orientation", y="error_smooth",
                hue="Mapping", hue_order=HUE_ORDER, palette=PALETTE,
                kind="line", col="subject", col_wrap=4, height=2.8)
g.map(plt.axhline, y=0, color="k", linestyle="--", alpha=0.5)
for ori in (45, 90, 135):
    g.map(plt.axvline, x=ori, color="grey", linestyle=":", alpha=0.5)
g.set_axis_labels("Orientation (°)", "Mean signed error (CHF)")
g.figure.suptitle("Per-subject signed error by orientation", y=1.03)

In [ ]:
import pingouin as pg

In [ ]:
err['orientation_bin'] = pd.cut(err['orientation'], bins=[0.0, 45.0, 90.0, 135.0, 180.0])

tmp = err.groupby(['subject', 'Mapping', 'orientation_bin']).mean()

tmp.groupby(['Mapping', 'orientation_bin']).apply(lambda d: pg.ttest(d['error'], 0))

### Composite test: is the CDF bin pattern, as a whole, the mirror image of Inverse CDF?

Each of the four per-bin tests above is individually underpowered (n≈27-28, single bin). Since `cdf` and `inverse_cdf` are reflections of each other, the predicted CDF bias in each bin is the *opposite sign* of the observed group-level Inverse CDF bias in that same bin. We align each CDF subject's per-bin error to that predicted sign, average the four sign-aligned bins within subject (one composite score per CDF subject), and run a single one-sample t-test (one-sided: composite > 0) — a much better-powered test of the overall directional prediction than eyeballing four separate bins.

In [ ]:
# Sign-align each CDF subject's per-bin bias to the (opposite) Inverse CDF group pattern,
# then test the within-subject average against 0 (one-sided, predicted direction).
cdf_err = tmp.xs('CDF', level='Mapping')['error'].unstack('orientation_bin')
inv_err = tmp.xs('Inverse CDF', level='Mapping')['error'].unstack('orientation_bin')

predicted_sign = -np.sign(inv_err.mean(axis=0))  # per-bin, from the Inverse CDF group mean
composite = cdf_err.mul(predicted_sign, axis=1).mean(axis=1)  # one score per CDF subject

print('Predicted sign per bin (mirror of Inverse CDF):')
print(predicted_sign)
print()
pg.ttest(composite.dropna(), 0, alternative='greater')

## Preregistered valuation-bias test: efficient coding at the valuation stage

This is the Bedi et al. (2026) preregistered confirmatory analysis (`notes/papers/bedi_et_al2026.pdf`),
run on this cohort. Valuation bias is `bias = bid - true value`, regressed on orientation
centred at the cardinal (`theta_c = orientation - 90`) inside the preregistered window
**65-115 deg** -- chosen because that is where the competing accounts separate, and it stays
clear of the value-mapping boundaries.

The diagnostic is the **sign of the bias slope, and whether it reverses between mappings**.
Perceptual input statistics are identical across conditions; only the stimulus-value mapping
changes, so only the *induced value prior* differs. An efficient code in perception alone
cannot flip the sign of the bias pattern -- it would just add small Jacobian effects. A
**reversal between Aligned and Misaligned is therefore the signature of efficient coding in
value space** (paper Fig. 3e), i.e. at the second, valuation stage: the value representation
re-allocates its resources to the short-term value prior induced by the current mapping, and
Bayesian decoding of that efficient code repels estimates away from the dense region.

**Condition naming.** This repo's `mapping` values follow the *construction*: `cdf` is the
scaled-CDF transform of the long-term orientation prior = the paper's **Misaligned**;
`inverse_cdf` is its inverse = the paper's **Aligned**. That also matches the paper's own
caption logic -- value density sits at the obliques under `cdf` and at the cardinal under
`inverse_cdf`. (Note the columns of the paper's Supplementary Table 1 are headed the other
way round; the prose, the construction and the slopes below all agree with the mapping used
here.) Published betas for comparison: Aligned +0.0574, Misaligned -0.0257, divergence
0.0831 (119 participants, Phase 3).

In [ ]:
from scipy import stats
import statsmodels.formula.api as smf

ORI_WINDOW = (65, 115)     # preregistered window around the cardinal
PAPER_NAME = {"cdf": "Misaligned", "inverse_cdf": "Aligned"}
COND_ORDER = ["Aligned", "Misaligned"]
COND_PALETTE = {"Aligned": PALETTE["Inverse CDF"], "Misaligned": PALETTE["CDF"]}

bias_df = df[df["orientation"].between(*ORI_WINDOW)].dropna(subset=["error"]).copy()
bias_df["theta_c"] = bias_df["orientation"] - 90.0
bias_df["Condition"] = pd.Categorical(bias_df["mapping"].map(PAPER_NAME),
                                      categories=["Misaligned", "Aligned"])

print(f"Window {ORI_WINDOW[0]}-{ORI_WINDOW[1]} deg: {len(bias_df)} bids, "
      f"{bias_df['subject'].nunique()} participants")
print()

# Preregistered model: bias ~ theta_c * mapping + (1 | participant)
full = smf.mixedlm("error ~ theta_c * Condition", bias_df,
                   groups=bias_df["subject"], re_formula="1").fit()
print(full.summary().tables[1])
print()

# Within-condition slopes, against the published estimates
PUBLISHED = {"Aligned": 0.0574, "Misaligned": -0.0257}
for cond in COND_ORDER:
    sub = bias_df[bias_df["Condition"] == cond]
    m = smf.mixedlm("error ~ theta_c", sub, groups=sub["subject"], re_formula="1").fit(method="lbfgs")
    b, se = m.params["theta_c"], m.bse["theta_c"]
    print(f"{cond:<11} beta = {b:+.5f} +/- {se:.5f}, t = {b / se:6.2f}, "
          f"p = {m.pvalues['theta_c']:.2e}   (published {PUBLISHED[cond]:+.4f})"
          f"{'' if m.converged else '   [!] did not converge'}")

div = full.params["theta_c:Condition[T.Aligned]"]
div_se = full.bse["theta_c:Condition[T.Aligned]"]
print(f"{'Divergence':<11} beta = {div:+.5f} +/- {div_se:.5f}, t = {div / div_se:6.2f}"
      f"   (published +0.0831)")
print()

# Assumption-light check: the same slopes as a plain per-subject OLS, averaged
subj = (bias_df.groupby(["subject", "Condition"], observed=True)
               .apply(lambda g: stats.linregress(g["theta_c"], g["error"]).slope)
               .rename("slope").reset_index())
for cond in COND_ORDER:
    v = subj.loc[subj["Condition"] == cond, "slope"]
    print(f"{cond:<11} per-subject mean {v.mean():+.5f} +/- {v.sem():.5f} (n = {len(v)})")
paired = subj.pivot(index="subject", columns="Condition", values="slope").dropna()
print(pg.ttest(paired["Aligned"], paired["Misaligned"], paired=True).round(4).to_string())

In [ ]:
# Group-level bias around the cardinal: the slope reverses with the value prior
cell = (bias_df.groupby(["subject", "Condition", "orientation"], observed=True)["error"]
               .mean().reset_index())

with sns.plotting_context("talk"), sns.axes_style("ticks"):
    fig, ax = plt.subplots(figsize=(5.6, 4.2), constrained_layout=True)
    ax.axhline(0, color="0.6", ls="--", lw=1.0, zorder=0)

    for cond in COND_ORDER:
        d = cell[cell["Condition"] == cond]
        m = d.groupby("orientation")["error"].agg(["mean", "sem"])
        ax.fill_between(m.index, m["mean"] - m["sem"], m["mean"] + m["sem"],
                        color=COND_PALETTE[cond], alpha=0.22, lw=0, zorder=1)
        ax.plot(m.index, m["mean"], "o-", color=COND_PALETTE[cond],
                ms=5, lw=1.6, zorder=2)
        ax.text(m.index[-1] + 1.5, m["mean"].iloc[-1], cond, color=COND_PALETTE[cond],
                fontsize=12, ha="left", va="center")

    ax.annotate("Repelled from the\ndense value region",
                xy=(75, cell[(cell["Condition"] == "Aligned") &
                             (cell["orientation"] == 75)]["error"].mean()),
                xytext=(69, -2.4), fontsize=11, color=COND_PALETTE["Aligned"],
                ha="left", va="center",
                arrowprops=dict(arrowstyle="-|>", color=COND_PALETTE["Aligned"], lw=1.1,
                                mutation_scale=9, shrinkA=3, shrinkB=8,
                                relpos=(0.5, 1.0)))

    ax.set_xticks([67.5, 90, 112.5])
    ax.set_xlim(64, 124)
    ax.set_xlabel("Orientation (deg)")
    ax.set_ylabel("Valuation bias (CHF)")
    ax.set_title("Bias around the cardinal", pad=10)
    sns.despine(offset=5, trim=True, ax=ax)

## Variability of responses by orientation

Within-subject SD of the response per orientation bin (smoothed with a centered 3-bin rolling window). Shown both in raw CHF and z-scored within subject — raw shows absolute precision differences across participants; z-scored isolates the orientation-shape independent of each participant's overall noise level.

In [ ]:
var = (df.groupby(["subject", "Mapping", "orientation"], as_index=False)["response"]
          .std()
          .rename(columns={"response": "response_std"})
          .sort_values(["subject", "Mapping", "orientation"]))
var["response_std_smooth"] = (var.groupby(["subject", "Mapping"])["response_std"]
                              .transform(lambda s: s.rolling(3, center=True, min_periods=1).mean()))
var["response_std_z"] = (var.groupby("subject")["response_std_smooth"]
                         .transform(lambda s: (s - s.mean()) / s.std()))

# Raw SD in CHF
g = sns.relplot(data=var, x="orientation", y="response_std_smooth",
                hue="Mapping", hue_order=HUE_ORDER, palette=PALETTE,
                kind="line", errorbar="se", height=4, aspect=1.4)
g.set_axis_labels("Orientation (°)", "SD of response (CHF)")
plt.axhline(0, color="k", linestyle="--", alpha=0.5)
for ori in (45, 90, 135):
    plt.axvline(ori, color="grey", linestyle=":", alpha=0.5)
g.figure.suptitle("Response variability by orientation (raw)", y=1.02)

# Z-scored within subject
g = sns.relplot(data=var, x="orientation", y="response_std_z",
                hue="Mapping", hue_order=HUE_ORDER, palette=PALETTE,
                kind="line", errorbar="se", height=4, aspect=1.4)
g.set_axis_labels("Orientation (°)", "SD of response (z within subject)")
plt.axhline(0, color="k", linestyle="--", alpha=0.5)
for ori in (45, 90, 135):
    plt.axvline(ori, color="grey", linestyle=":", alpha=0.5)
g.figure.suptitle("Response variability by orientation (z-scored within subject)", y=1.02)

## Subjectwise variability of responses

Within-subject SD of the response (smoothed over a centered 3-bin rolling window), one panel per subject, both conditions overlaid. Shown as a function of **orientation** (stimulus axis) and of the **true value n** (CHF) — the latter puts both conditions on a common magnitude axis, so it probes scalar variability (does response noise grow with n?). A group-level summary across subjects as a function of value follows.

In [ ]:
# Within-subject SD of response, per orientation bin and per value (n) bin
def _sd_by(group_col):
    out = (df.groupby(["subject", "Mapping", group_col], as_index=False)["response"]
              .std()
              .rename(columns={"response": "response_std"})
              .sort_values(["subject", "Mapping", group_col]))
    out["response_std_smooth"] = (out.groupby(["subject", "Mapping"])["response_std"]
                                  .transform(lambda s: s.rolling(3, center=True, min_periods=1).mean()))
    return out

var_ori = _sd_by("orientation")
var_n   = _sd_by("value")

# Per-subject panels — as a function of orientation
g = sns.relplot(data=var_ori, x="orientation", y="response_std_smooth",
                hue="Mapping", hue_order=HUE_ORDER, palette=PALETTE,
                kind="line", col="subject", col_wrap=4, height=2.8)
for ax in g.axes.flat:
    for ori in (45, 90, 135):
        ax.axvline(ori, color="grey", linestyle=":", alpha=0.5)
g.set_axis_labels("Orientation (°)", "SD of response (CHF)")
g.set(xlim=ORI_RANGE, ylim=(0, None), xticks=[0, 45, 90, 135, 180])
g.figure.suptitle("Per-subject response variability across orientation", y=1.03)

# Per-subject panels — as a function of value (n)
g = sns.relplot(data=var_n, x="value", y="response_std_smooth",
                hue="Mapping", hue_order=HUE_ORDER, palette=PALETTE,
                kind="line", marker="o", col="subject", col_wrap=4, height=2.8)
g.set_axis_labels("Value n (CHF)", "SD of response (CHF)")
g.set(xlim=VALUE_RANGE, ylim=(0, None), xticks=[0, 10, 20, 30, 42])
g.figure.suptitle("Per-subject response variability across value (n)", y=1.03)

# Group-level summary across subjects — as a function of value (n)
g = sns.relplot(data=var_n, x="value", y="response_std_smooth",
                hue="Mapping", hue_order=HUE_ORDER, palette=PALETTE,
                kind="line", marker="o", errorbar="se", height=4, aspect=1.4)
g.set_axis_labels("Value n (CHF)", "SD of response (CHF)")
g.set(xlim=VALUE_RANGE, ylim=(0, None), xticks=[0, 10, 20, 30, 42])
g.figure.suptitle("Response variability across value (n), across subjects", y=1.02)

## Response time across runs (fatigue / learning)

In [ ]:
rt_run = (df.groupby(["subject", "Mapping", "run"], as_index=False)["rt"]
            .mean())

g = sns.relplot(data=rt_run, x="run", y="rt",
                hue="Mapping", hue_order=HUE_ORDER, palette=PALETTE,
                kind="line", errorbar="se", height=4, aspect=1.4)
g.set_axis_labels("Run", "Mean RT (s)")
g.figure.suptitle("RT across runs", y=1.02)

## RT by orientation / magnitude (group level)

Group-level RT as a function of orientation and of the true value n (CHF), per mapping condition. Per subject we take the **median** RT within each bin (robust to RT skew), then average those subject medians across subjects (smoothed with a centered 3-bin rolling window). Shaded bands are ±1 SEM across subjects.

In [ ]:
# Per-subject median RT within each bin, then smoothed -> group mean ±SEM across subjects
def _median_rt_by(group_col):
    out = (df.groupby(["subject", "Mapping", group_col], as_index=False)["rt"]
              .median()
              .sort_values(["subject", "Mapping", group_col]))
    out["rt_smooth"] = (out.groupby(["subject", "Mapping"])["rt"]
                        .transform(lambda s: s.rolling(3, center=True, min_periods=1).mean()))
    return out

rt_ori = _median_rt_by("orientation")
rt_n   = _median_rt_by("value")

# RT by orientation
g = sns.relplot(data=rt_ori, x="orientation", y="rt_smooth",
                hue="Mapping", hue_order=HUE_ORDER, palette=PALETTE,
                kind="line", errorbar="se", height=4, aspect=1.4)
for ori in (45, 90, 135):
    plt.axvline(ori, color="grey", linestyle=":", alpha=0.5)
g.set_axis_labels("Orientation (°)", "Median RT (s)")
g.set(xlim=ORI_RANGE, xticks=[0, 45, 90, 135, 180])
g.figure.suptitle("RT by orientation (across subjects)", y=1.02)

# RT by value (n)
g = sns.relplot(data=rt_n, x="value", y="rt_smooth",
                hue="Mapping", hue_order=HUE_ORDER, palette=PALETTE,
                kind="line", marker="o", errorbar="se", height=4, aspect=1.4)
g.set_axis_labels("Value n (CHF)", "Median RT (s)")
g.set(xlim=VALUE_RANGE, xticks=[0, 10, 20, 30, 42])
g.figure.suptitle("RT by value (n) (across subjects)", y=1.02)

## Response error by acquisition date (trackball QC)

Per-run stats plotted against the run's acquisition timestamp (file mtime of the run's `events.tsv`, which survives the sourcedata → BIDS → cluster rsync chain intact), split by mapping condition. A single "mean error" trend can hide what's actually going on, since positive and negative per-run biases from different subjects/days can cancel in a pooled linear fit. Three panels separate this out:

- **Bias** — per-run mean signed error (`response - value`). Directional drift (e.g. a trackball that systematically pulls one way).
- **|Bias|** — magnitude of the same per-run mean error, sign discarded. Catches growing miscalibration even if its direction varies run to run (which a signed trend would average out).
- **Noise** — within-run SD of the trial-level signed error. Precision/consistency, independent of any directional bias.

A rising trend in **|Bias|** or **Noise** over calendar time would be consistent with a degrading response device.


In [ ]:
import matplotlib.dates as mdates
from scipy import stats

run_date = (df.groupby(["subject", "session", "run", "Mapping"], as_index=False)
              .agg(run_datetime=("run_datetime", "first"),
                   bias=("error", "mean"),
                   noise=("error", "std"))
              .sort_values("run_datetime"))
run_date["abs_bias"] = run_date["bias"].abs()
run_date["day_ordinal"] = run_date["run_datetime"].map(lambda d: d.toordinal() + d.hour / 24 + d.minute / 1440)

metrics = [("bias", "Bias (CHF)"),
           ("abs_bias", "|Bias| (CHF)"),
           ("noise", "Noise, within-run SD (CHF)")]

with sns.plotting_context("talk"), sns.axes_style("ticks"):
    fig, axes = plt.subplots(1, 3, figsize=(15.5, 4.3), constrained_layout=True, sharex=True)
    for ax, (col, label) in zip(axes, metrics):
        for i, m in enumerate(HUE_ORDER):
            sub = run_date[run_date["Mapping"] == m]
            ax.scatter(sub["run_datetime"], sub[col], s=20, alpha=0.5,
                       color=PALETTE[m], edgecolor="white", linewidth=0.3)

            slope, intercept, r, p, se = stats.linregress(sub["day_ordinal"], sub[col])
            xs = np.linspace(sub["day_ordinal"].min(), sub["day_ordinal"].max(), 100)
            fit_dates = [pd.Timestamp.fromordinal(int(x)) for x in xs]
            ax.plot(fit_dates, slope * xs + intercept, color=PALETTE[m], lw=1.6)

            ax.annotate(f"{m}: {slope:+.4f}/day (p={p:.3f})", xy=(0.03, 0.96 - 0.09 * i),
                        xycoords="axes fraction", ha="left", va="top", fontsize=9.5,
                        color=PALETTE[m], fontweight="bold")
        ax.set_ylabel(label)
        ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(ax.xaxis.get_major_locator()))
        sns.despine(ax=ax, offset=5, trim=True)

    axes[0].axhline(0, color="0.7", lw=0.8, ls="--", zorder=0)
    fig.suptitle("Response error by acquisition date, split by mapping condition", y=1.05)

    fig.savefig("../notes/figures/error_by_acquisition_date.pdf", bbox_inches="tight", pad_inches=0.02)


## Calibration over the course of the study (early / mid / late thirds)

Mean estimate as a function of orientation, split into three equal-sized chunks of the study timeline (by acquisition date), veridical (true) mapping overlaid as dashed lines. If the trackball's growing noise (previous section) is degrading actual calibration — not just adding scatter — the solid response curves should drift away from the dashed veridical lines in the later panels.


In [ ]:
df["time_bin"] = pd.qcut(df["run_datetime"], 3, labels=["Early", "Mid", "Late"])
bin_ranges = df.groupby("time_bin", observed=True)["run_datetime"].agg(["min", "max"])
bin_labels = {b: f"{r['min']:%b %d} - {r['max']:%b %d}" for b, r in bin_ranges.iterrows()}
df["time_bin_label"] = df["time_bin"].map(bin_labels)
time_bin_order = [bin_labels[b] for b in ["Early", "Mid", "Late"]]

mean_resp_time = (df.groupby(["subject", "Mapping", "time_bin_label", "orientation"], as_index=False)["response"]
                     .mean())

def _panel(data, **kwargs):
    ax = plt.gca()
    ax.axline((ORI_RANGE[0], VALUE_RANGE[0]), (ORI_RANGE[1], VALUE_RANGE[1]),
              color="k", ls=":", lw=0.8, alpha=0.5, zorder=-1)
    for m in HUE_ORDER:
        sub = truth[truth["Mapping"] == m].sort_values("orientation")
        ax.plot(sub["orientation"], sub["value"], color=PALETTE[m], ls="--", lw=1.2, alpha=0.5, zorder=0)
    sns.lineplot(data=data, x="orientation", y="response", hue="Mapping", hue_order=HUE_ORDER,
                palette=PALETTE, errorbar=("se", 1), legend=False, ax=ax)

with sns.plotting_context("talk"), sns.axes_style("ticks"):
    g = sns.FacetGrid(mean_resp_time, col="time_bin_label", col_order=time_bin_order,
                      height=3.6, aspect=1.1, despine=False)
    g.map_dataframe(_panel)
    g.set_titles("{col_name}")
    g.set_axis_labels("Orientation (°)", "Mean estimate (CHF)")
    for ax in g.axes.flat:
        ax.set_xlim(0, 210)
        ax.set_ylim(VALUE_RANGE)
        ax.set_xticks([0, 45, 90, 135, 180])
        ax.set_yticks([0, 10, 20, 30, 42])
    sns.despine(fig=g.figure, offset=5, trim=True)
    g.figure.suptitle("Mean estimate vs orientation, early / mid / late thirds of the study", y=1.05)

    ax_last = g.axes.flat[-1]
    endpoints = (mean_resp_time[(mean_resp_time["time_bin_label"] == time_bin_order[-1]) &
                                 (mean_resp_time["orientation"] == mean_resp_time["orientation"].max())]
                 .groupby("Mapping")["response"].mean())
    x_label = mean_resp_time["orientation"].max() + 4
    for m in HUE_ORDER:
        ax_last.text(x_label, endpoints[m], m, color=PALETTE[m], fontsize=10.5,
                     ha="left", va="center", fontweight="bold")

    g.figure.savefig("../notes/figures/calibration_by_time_bin.pdf", bbox_inches="tight", pad_inches=0.02)
